In [0]:
from pyspark.sql.functions import col, current_date, lit, current_timestamp

spark.sql("CREATE SCHEMA IF NOT EXISTS nyc311_project.gold")

silver = spark.table("nyc311_project.silver.service_requests")

# Build the initial SCD2 snapshot: one row per complaint, marked as the current version
dim_complaint_status = (silver
    .select("unique_key", "status", "agency", "complaint_type", "created_ts")
    .withColumn("valid_from", current_date())
    .withColumn("valid_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("_scd_loaded_at", current_timestamp())
)

dim_complaint_status.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc311_project.gold.dim_complaint_status")

print(f"Initial SCD2 rows: {dim_complaint_status.count()}")
display(dim_complaint_status.limit(10))

Initial SCD2 rows: 199053


unique_key,status,agency,complaint_type,created_ts,valid_from,valid_to,is_current,_scd_loaded_at
61659328,Closed,NYPD,Blocked Driveway,2024-06-30T23:59:55.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z
61656368,Closed,NYPD,Noise - Residential,2024-06-30T23:59:54.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z
61661023,Closed,NYPD,Noise - Residential,2024-06-30T23:59:49.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z
61660929,Closed,NYPD,Noise - Commercial,2024-06-30T23:59:36.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z
61653851,Closed,NYPD,Noise - Commercial,2024-06-30T23:59:19.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z
61660457,Closed,NYPD,Illegal Parking,2024-06-30T23:59:17.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z
61653990,Closed,NYPD,Noise - Street/sidewalk,2024-06-30T23:58:55.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z
61654355,Closed,NYPD,Illegal Parking,2024-06-30T23:58:41.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z
61656072,Closed,NYPD,Disorderly Youth,2024-06-30T23:58:31.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z
61658314,Closed,NYPD,Illegal Fireworks,2024-06-30T23:57:49.000Z,2026-10-06,null,true,2026-10-06T06:07:47.554Z


In [0]:
dim_agency = silver.select("agency", "agency_name").distinct()
dim_agency.write.format("delta").mode("overwrite").saveAsTable("nyc311_project.gold.dim_agency")

dim_location = silver.select("unique_key", "borough", "incident_zip", "city", "latitude", "longitude").distinct()
dim_location.write.format("delta").mode("overwrite").saveAsTable("nyc311_project.gold.dim_location")

fact_requests = silver.select(
    "unique_key", "agency", "complaint_type", "descriptor", "status",
    "created_ts", "closed_ts", "borough", "incident_zip"
)
fact_requests.write.format("delta").mode("overwrite").saveAsTable("nyc311_project.gold.fact_requests")

print("Gold tables built.")

Gold tables built.
